In [1]:
!pip uninstall crewai -y
!pip install crewai --no-cache-dir

INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 13.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 MB 15.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 15.0 MB/s eta 0:00:00 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.3 MB/s eta 0:00:00
   ━━━━━━

In [21]:
# Let's import crewAI
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool

In [12]:
import warnings
warnings.filterwarnings('ignore')
!pip install --upgrade pip
!pip install -q crewai python-dotenv langchain_openai pandas 

In [30]:
# Import necessary libraries
import os
import pandas as pd

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown, Image

# Load environment variables (especially API keys)
load_dotenv()

# Configure API Key, which is essential for AI agents to use OpenAI models
openai_api_key = os.getenv("OPENAI_API_KEY")

# Initialize the LLM for the agents
llm = LLM(model="gpt-4.1-mini-2025-04-14", api_key = openai_api_key)



In [31]:
# Helper function to print markdowns
def print_markdown(text):
    """Displays text as Markdown in Jupyter."""
    display(Markdown(text))

In [32]:
from notebookExecutor import NotebookCodeExecutor, NotebookCodeExecutorSchema

In [33]:
# First, let's load the data into a shared DataFrame that our tool can access
# This simulates the data being available in the execution environment
import pandas as pd
file_path = "Supplement_Sales_Weekly_Expanded.csv"
shared_df = pd.read_csv(file_path)

In [34]:
# Let's instantiate the custom tool
notebook_executor_tool = NotebookCodeExecutor(namespace=globals())
print("✅ Custom tool 'NotebookCodeExecutor' instantiated with notebook's global namespace.")

✅ Custom tool 'NotebookCodeExecutor' instantiated with notebook's global namespace.


In [35]:
# Let's test the Notebook Code executor tool 
# Let's define a basic function that adds two numbers
def add_numbers(a,b):
    return a + b

In [36]:
# Test the tool
test_code = "print(add_numbers(1,2))"
print("\nTesting tool:\n")
print(notebook_executor_tool.run(code = test_code))



Testing tool:

--- Executing Code ---
Code executed successfully. Output:
```output
3

```



In [37]:
# Define the Data Science Planner Agent (no tool needed)
planner_agent = Agent(role = "Lead Data Scientist and Planner",
                      goal = ("Analyze the objective (predict 'Units Sold') assuming data is in a global pandas DataFrame 'shared_df'. "
                              "Create a step-by-step plan for regression analysis. Instruct subsequent agents on the GOALS for each step."
                              "(e.g., inspect data, preprocess, model, evaluate) and tell them to use the 'Notebook Code Executor' tool "
                              "to WRITE and EXECUTE the necessary Python code."),
                backstory = ("Experienced data scientist planning ML projects. Knows data is in 'shared_df' and agents will write and execute code using a tool."),
                llm = llm,
                    allow_delegation = False,
                    verbose = True)

In [38]:
# Define the Data Analysis and Preprocessing Agent (needs access to notebook_executer_tool to generate code)
analyst_preprocessor_agent = Agent(role = "Data Analysis and Preprocessing Expert",
                                   goal = (
        "Follow the plan for data analysis and preprocessing. **Write the necessary Python code** using pandas and scikit-learn "
        "to operate on the global pandas DataFrame 'shared_df'. Your code must perform inspection (shape, info, nulls, describe), "
        "handle date/identifiers (convert 'Date', sort, drop 'Date'/'Product Name'), encode categoricals (OneHotEncode 'Platform' modifying 'shared_df'), "
        "and finally **create the global variables X_train, X_test, y_train, y_test** from 'shared_df' using an 80/20 split (shuffle=False). "
        "Use the 'Notebook Code Executor' tool to execute the code you write. Ensure your generated code includes print statements for key results."),
    
                                   backstory = (
        "Meticulous analyst skilled in writing pandas/sklearn code. Uses the 'Notebook Code Executor' tool to run the generated code. "
        "Knows data is in global 'shared_df' and must create global train/test variables."),
                                   llm = llm,
                                   tools = [notebook_executor_tool],  # Assign the custom tool explicitly
                                   allow_delegation = False,
                                   verbose = True)


In [39]:
# Define the Modeling and Evaluation Agent (needs access to notebook_executer_tool to generate code)
modeler_evaluator_agent = Agent(role = "Machine Learning Modeler and Evaluator",
                                goal = (
        "Follow the plan for modeling and evaluation. **Write the necessary Python code** using scikit-learn. "
        "Assume global variables X_train, X_test, y_train, y_test exist. Your code must train a RandomForestRegressor(random_state=42), "
        "make predictions on X_test, calculate and print evaluation metrics (MAE, MSE, RMSE, R²), and print the top 10 feature importances. "
        "Use the 'Notebook Code Executor' tool to execute the code you write. "
        "Finally, include the exact Python code you generated and executed in your final response, formatted in a markdown block."
    ),
                                backstory = (
        "ML engineer specialized in regression. Writes scikit-learn code and uses the 'Notebook Code Executor' tool to run it. "
        "Expects global train/test split variables (X_train etc.) to be available."
    ),
    llm = llm,
    tools = [notebook_executor_tool],  # Assign the custom tool explicitly
    allow_delegation = False,
    verbose = True)


In [40]:
print("✅ CrewAI Agents defined, focusing on code generation.")
print(f"- {planner_agent.role}")
print(f"- {analyst_preprocessor_agent.role} (Tool: {analyst_preprocessor_agent.tools[0].name})")
print(f"- {modeler_evaluator_agent.role} (Tool: {modeler_evaluator_agent.tools[0].name})")

✅ CrewAI Agents defined, focusing on code generation.
- Lead Data Scientist and Planner
- Data Analysis and Preprocessing Expert (Tool: Notebook Code Executor)
- Machine Learning Modeler and Evaluator (Tool: Notebook Code Executor)


In [41]:
# Define the Planning Task (Stays largely the same, instructs agents on GOALS)
planning_task = Task(
    description = (
        "1. Goal: Create a plan for regression predicting 'Units Sold'.\n"
        "2. Data Context: Global pandas DataFrame 'shared_df' is available.\n"
        "3. Plan Steps: Outline sequence, instructing agents on their GOALS for each step and to use the 'Notebook Code Executor' tool to WRITE and RUN Python code:\n"
        "    a. Goal: Inspect global 'shared_df' (shape, info, nulls, describe).\n"
        "    b. Goal: Preprocess global 'shared_df' (handle Date [to_datetime, sort, drop], drop identifiers ['Product Name'], OneHotEncode 'Platform' [update 'shared_df'], create global X/y vars, create global train/test split vars X_train/test, y_train/test [80/20, shuffle=False]).\n"
        "    c. Goal: Train RandomForestRegressor using global X_train, y_train (use random_state=42).\n"
        "    d. Goal: Evaluate model on global X_test (predict, calc & print MAE, MSE, RMSE, R2).\n"
        "    e. Goal: Extract & print top 10 feature importances from the trained model.\n"
        "5. Output: Numbered plan focusing on the objectives for each data science step."
    ),
    expected_output = (
        "Numbered plan outlining the data science goals for subsequent agents, reminding them to generate code and use the 'Notebook Code Executor' tool, interacting with global variables like 'shared_df' and 'X_train'."
    ),
    agent = planner_agent)


In [42]:
# Define the Data Analysis and Preprocessing Task (High-level instructions)
data_analysis_preprocessing_task = Task(
    description = (
        "Follow the analysis/preprocessing plan. Your goal is to inspect and prepare the global 'shared_df' DataFrame and create global training/testing variables. "
        "You MUST **generate Python code** to achieve this and then execute it using the 'Notebook Code Executor' tool. "
        "Specifically, your generated code needs to:\n"
        "1. Inspect the 'shared_df' DataFrame (print shape, info(), isnull().sum(), describe()).\n"
        "2. Convert 'Date' column in 'shared_df' to datetime objects, sort 'shared_df' by 'Date', then drop the 'Date' and 'Product Name' columns from 'shared_df'.\n"
        "3. One-Hot Encode the 'Platform' column in 'shared_df' (use pd.get_dummies, drop_first=True). **Crucially, ensure 'shared_df' DataFrame variable is updated with the result of the encoding.**\n"
        "4. Create a global variable 'y' containing the 'Units Sold' column from 'shared_df'.\n"
        "5. Create a global variable 'X' containing the remaining columns from the updated 'shared_df' (after dropping 'Units Sold').\n"
        "6. Split 'X' and 'y' into global variables: 'X_train', 'X_test', 'y_train', 'y_test' using an 80/20 split with `shuffle=False`. Ensure these four variables are created in the global scope.\n"
        "Make sure your generated code includes necessary imports (like pandas, train_test_split) and print statements for verification (e.g., printing shapes of created variables like X_train.shape)."
        # "Remember to pass the required libraries (e.g., ['pandas', 'scikit-learn']) to the tool if your code uses them, although they should be pre-imported in this notebook." # Optional hint, often the agent figures out imports
    ),
    expected_output = (
        "Output from the 'Notebook Code Executor' tool showing the successful execution of agent-generated code. This includes printouts confirming:\n"
        "- Initial data inspection results for 'shared_df'.\n"
        "- Confirmation of DataFrame modifications (e.g., shape after encoding).\n"
        "- Confirmation of the creation and shapes of global variables X, y, X_train, X_test, y_train, y_test."
    ),
    agent = analyst_preprocessor_agent,
    tools = [notebook_executor_tool],  # Explicitly list tool
)


In [43]:
# Define the Modeling and Evaluation Task (High-level instructions)
modeling_evaluation_task = Task(
    description = (
        "Follow the modeling/evaluation plan. Your goal is to train a model, evaluate it, and report results. "
        "You MUST **generate Python code** assuming global variables X_train, X_test, y_train, y_test exist, and execute it using the 'Notebook Code Executor' tool. "
        "Specifically, your generated code needs to:\n"
        "1. Train a `RandomForestRegressor` model (use `random_state=42`) using the global `X_train` and `y_train` variables. Store the trained model in a global variable named `trained_model`.\n"
        "2. Make predictions on the global `X_test` variable.\n"
        "3. Calculate and print the MAE, MSE, RMSE, and R-squared metrics by comparing predictions against the global `y_test` variable.\n"
        "4. Calculate and print the top 10 feature importances from the trained model (using `X_train.columns` for feature names).\n"
        "Make sure your generated code includes necessary imports (like RandomForestRegressor, metrics functions from sklearn.metrics, numpy, pandas) and print statements for all results.\n"
        "Finally, include the exact Python code you generated and executed within a markdown code block (```python...```) in your final response."
        # "Remember to pass required libraries like ['scikit-learn', 'pandas', 'numpy'] to the tool if needed." # Optional hint
    ),
    expected_output = (
        "Output from the 'Notebook Code Executor' tool showing the successful execution of agent-generated code, including:\n"
        "- Printed regression metrics (MAE, MSE, RMSE, R²).\n"
        "- Printed top 10 feature importances.\n"
        "The final response MUST also contain a markdown code block (```python...```) showing the exact Python code that was generated and executed for these steps."
    ),
    agent = modeler_evaluator_agent,
    tools = [notebook_executor_tool],  # Explicitly list tool
)

print("✅ CrewAI Tasks defined with high-level instructions for code generation.")

✅ CrewAI Tasks defined with high-level instructions for code generation.


In [44]:
# Let's Create the Crew
regression_crew = Crew(
    agents = [planner_agent, analyst_preprocessor_agent, modeler_evaluator_agent],
    tasks = [planning_task, data_analysis_preprocessing_task, modeling_evaluation_task],
    process = Process.sequential,
    verbose = 1,  # Use detailed output to see agent thoughts and tool usage
    output_log_file = True)

In [45]:
# Kick off the crew execution!
print("Starting the Crew execution (Agents will generate code)...")

# Ensure the initial DataFrame exists before starting
crew_result = regression_crew.kickoff()

Starting the Crew execution (Agents will generate code)...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 4eb190e4-94a1-4672-ac6e-fcbc3d7ff746                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Goal: Create a plan for regression predicting 'Units Sold'.                                           │
│  2. Data Context: Global pandas DataFrame 'shared_df' is available.                                             │
│  3. Plan Steps: Outline sequence, instructing agents on their GOALS for each step and to use the 'Notebook      │
│  Code Executor' tool to WRITE and RUN Python code:                                                              │
│      a. Goal: Inspect global 'shared_df' (shape, info, nulls, describe).                                        │
│      b. Goal: Preprocess global 'shared_df' (handle Date [to_datetime, sort, drop], drop identifiers ['Product  │
│  Name'], OneHotEncode 'Platform' [update 'shared_df'], create global X/y vars, create global train/test split   │
│  vars X_train/test, y_train/test [80/20, shuffle=False]).                                                       │
│      c. Goal: Train RandomForestRegressor using global X_train, y_train (use random_state=42).                  │
│      d. Goal: Evaluate model on global X_test (predict, calc & print MAE, MSE, RMSE, R2).                       │
│      e. Goal: Extract & print top 10 feature importances from the trained model.                                │
│  5. Output: Numbered plan focusing on the objectives for each data science step.                                │
│  ID: 8662b4be-b489-4d4f-8711-2e86ede10b35                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Data Scientist and Planner                                                                         │
│                                                                                                                 │
│  Task: 1. Goal: Create a plan for regression predicting 'Units Sold'.                                           │
│  2. Data Context: Global pandas DataFrame 'shared_df' is available.                                             │
│  3. Plan Steps: Outline sequence, instructing agents on their GOALS for each step and to use the 'Notebook      │
│  Code Executor' tool to WRITE and RUN Python code:                                                              │
│      a. Goal: Inspect global 'shared_df' (shape, info, nulls, describe).                                        │
│      b. Goal: Preprocess global 'shared_df' (handle Date [to_datetime, sort, drop], drop identifiers ['Product  │
│  Name'], OneHotEncode 'Platform' [update 'shared_df'], create global X/y vars, create global train/test split   │
│  vars X_train/test, y_train/test [80/20, shuffle=False]).                                                       │
│      c. Goal: Train RandomForestRegressor using global X_train, y_train (use random_state=42).                  │
│      d. Goal: Evaluate model on global X_test (predict, calc & print MAE, MSE, RMSE, R2).                       │
│      e. Goal: Extract & print top 10 feature importances from the trained model.                                │
│  5. Output: Numbered plan focusing on the objectives for each data science step.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Data Scientist and Planner                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. Goal: Inspect the global DataFrame 'shared_df' to understand its structure and content. Specifically,       │
│  agents should:                                                                                                 │
│     - Print the shape of 'shared_df'.                                                                           │
│     - Use `.info()` to review data types and non-null counts.                                                   │
│     - Check for missing values per column.                                                                      │
│     - Generate descriptive statistics with `.describe()`.                                                       │
│     Agents must WRITE and EXECUTE Python code using the 'Notebook Code Executor' tool to perform these          │
│  inspections and output the results.                                                                            │
│                                                                                                                 │
│  2. Goal: Preprocess the global DataFrame 'shared_df' preparing it for modeling. Agents should:                 │
│     - Convert the 'Date' column to datetime format, sort the DataFrame by 'Date', then drop the 'Date' column.  │
│     - Drop the 'Product Name' column (considered an identifier).                                                │
│     - OneHotEncode the 'Platform' column and update 'shared_df' accordingly.                                    │
│     - Define global feature matrix `X` (all columns except the target 'Units Sold') and target vector `y`       │
│  ('Units Sold').                                                                                                │
│     - Create train/test splits named `X_train`, `X_test`, `y_train`, and `y_test` using an 80/20 split. Set     │
│  `shuffle=False` to maintain time order.                                                                        │
│     All steps must be implemented by WRITING and EXECUTING Python code via the 'Notebook Code Executor' tool,   │
│  updating global variables as needed.                                                                           │
│                                                                                                                 │
│  3. Goal: Train a RandomForestRegressor model using the training data:                                          │
│     - Instantiate a RandomForestRegressor with `random_state=42`.                                               │
│     - Train the model on `X_train` and `y_train`.                                                               │
│     Agents should WRITE and EXECUTE the Python code using the 'Notebook Code Executor' tool, storing the        │
│  trained model in a global variable for later use.                                                              │
│                                                                                                                 │
│  4. Goal: Evaluate the trained model's performance on the test data:                                            │
│     - Use the trained model to predict 'Units Sold' for `X_test`.                                               │
│     - Calculate and PRINT the evaluation metrics: Mean Absolute Error (MAE), Mean Squared Error (MSE), Root     │
│  Mean Squared Error (RMSE), and R² score.              

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#4) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: notebook_code_executor                                                                                   │
│  Output: --- Executing Code ---                                                                                 │
│  Code executed successfully. Output:                                                                            │
│  ```output                                                                                                      │
│  Mean Absolute Error (MAE): 0.9399                                                                              │
│  Mean Squared Error (MSE): 2.6232                                                                               │
│  Root Mean Squared Error (RMSE): 1.6196                                                                         │
│  R-squared (R2): 0.9826                                                                                         │
│                                                                                                                 │
│  Top 10 Feature Importances:                                                                                    │
│  Revenue             0.529729                                                                                   │
│  Price               0.457497                                                                                   │
│  Discount            0.003526                                                                                   │
│  Units Returned      0.002405                                                                                   │
│  Location_USA        0.000817                                                                                   │
│  Platform_iHerb      0.000808                                                                                   │
│  Location_UK         0.000728                                                                                   │
│  Platform_Walmart    0.000720                                                                                   │
│  Category_Vitamin    0.000631                                                                                   │
│  Category_Mineral    0.000619                                                                                   │
│  dtype: float64                                                                                                 │
│                                                                                                                 │
│  ```                                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Machine Learning Modeler and Evaluator                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```python                                                                                                      │
│  from sklearn.ensemble import RandomForestRegressor                                                             │
│  from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score                                  │
│  import numpy as np                                                                                             │
│  import pandas as pd                                                                                            │
│  from sklearn.model_selection import train_test_split                                                           │
│                                                                                                                 │
│  # One-Hot Encode 'Category' and 'Location' as they were not encoded before                                     │
│  shared_df_encoded = shared_df.copy()                                                                           │
│  shared_df_encoded = pd.get_dummies(shared_df_encoded, columns=['Category', 'Location'], drop_first=True)       │
│                                                                                                                 │
│  # Define X and y                                                                                               │
│  y = shared_df_encoded['Units Sold']                                                                            │
│  X = shared_df_encoded.drop(columns=['Units Sold'])                                                             │
│                                                                                                                 │
│  # Split data into train/test sets 80/20 with shuffle=False                                                     │
│  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)                        │
│                                                                                                                 │
│  # Train the RandomForestRegressor                                                                              │
│  trained_model = RandomForestRegressor(random_state=42)                                                         │
│  trained_model.fit(X_train, y_train)                                                                            │
│                                                                                                                 │
│  # Predict on X_test                                                                                            │
│  predictions = trained_model.predict(X_test)                                                                    │
│                                                                                                                 │
│  # Calculate regression metrics                                                                                 │
│  mae = mean_absolute_error(y_test, predictions)                                                                 │
│  mse = mean_squared_error(y_test, predictions)                                                                  │
│  rmse = np.sqrt(mse)                                                                                            │
│  r2 = r2_score(y_test, predictions)                    

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Follow the modeling/evaluation plan. Your goal is to train a model, evaluate it, and report results.     │
│  You MUST **generate Python code** assuming global variables X_train, X_test, y_train, y_test exist, and        │
│  execute it using the 'Notebook Code Executor' tool. Specifically, your generated code needs to:                │
│  1. Train a `RandomForestRegressor` model (use `random_state=42`) using the global `X_train` and `y_train`      │
│  variables. Store the trained model in a global variable named `trained_model`.                                 │
│  2. Make predictions on the global `X_test` variable.                                                           │
│  3. Calculate and print the MAE, MSE, RMSE, and R-squared metrics by comparing predictions against the global   │
│  `y_test` variable.                                                                                             │
│  4. Calculate and print the top 10 feature importances from the trained model (using `X_train.columns` for      │
│  feature names).                                                                                                │
│  Make sure your generated code includes necessary imports (like RandomForestRegressor, metrics functions from   │
│  sklearn.metrics, numpy, pandas) and print statements for all results.                                          │
│  Finally, include the exact Python code you generated and executed within a markdown code block                 │
│  (```python...```) in your final response.                                                                      │
│  Agent: Machine Learning Modeler and Evaluator                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 4eb190e4-94a1-4672-ac6e-fcbc3d7ff746                                                                       │
│  Final Output: ```python                                                                                        │
│  from sklearn.ensemble import RandomForestRegressor                                                             │
│  from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score                                  │
│  import numpy as np                                                                                             │
│  import pandas as pd                                                                                            │
│  from sklearn.model_selection import train_test_split                                                           │
│                                                                                                                 │
│  # One-Hot Encode 'Category' and 'Location' as they were not encoded before                                     │
│  shared_df_encoded = shared_df.copy()                                                                           │
│  shared_df_encoded = pd.get_dummies(shared_df_encoded, columns=['Category', 'Location'], drop_first=True)       │
│                                                                                                                 │
│  # Define X and y                                                                                               │
│  y = shared_df_encoded['Units Sold']                                                                            │
│  X = shared_df_encoded.drop(columns=['Units Sold'])                                                             │
│                                                                                                                 │
│  # Split data into train/test sets 80/20 with shuffle=False                                                     │
│  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)                        │
│                                                                                                                 │
│  # Train the RandomForestRegressor                                                                              │
│  trained_model = RandomForestRegressor(random_state=42)                                                         │
│  trained_model.fit(X_train, y_train)                                                                            │
│                                                                                                                 │
│  # Predict on X_test                                                                                            │
│  predictions = trained_model.predict(X_test)                                                                    │
│                                                                                                                 │
│  # Calculate regression metrics                                                                                 │
│  mae = mean_absolute_error(y_test, predictions)                                                                 │
│  mse = mean_squared_error(y_test, predictions)                                                                  │
│  rmse = np.sqrt(mse)                                                                                            │
│  r2 = r2_score(y_test, predictions)                   

In [46]:
print("\n\n🏁 Crew execution finished.")
print("\nCrew Final Result (Output of last task):")
print("========================================")

print_markdown(crew_result.raw)



🏁 Crew execution finished.

Crew Final Result (Output of last task):


```python
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# One-Hot Encode 'Category' and 'Location' as they were not encoded before
shared_df_encoded = shared_df.copy()
shared_df_encoded = pd.get_dummies(shared_df_encoded, columns=['Category', 'Location'], drop_first=True)

# Define X and y
y = shared_df_encoded['Units Sold']
X = shared_df_encoded.drop(columns=['Units Sold'])

# Split data into train/test sets 80/20 with shuffle=False
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Train the RandomForestRegressor
trained_model = RandomForestRegressor(random_state=42)
trained_model.fit(X_train, y_train)

# Predict on X_test
predictions = trained_model.predict(X_test)

# Calculate regression metrics
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R2): {r2:.4f}")

# Calculate and print top 10 feature importances
feature_importances = pd.Series(trained_model.feature_importances_, index=X_train.columns)
top_10_features = feature_importances.nlargest(10)

print("\nTop 10 Feature Importances:")
print(top_10_features)
```